# 04 — Clean CPS Data

This notebook cleans the IPUMS CPS March ASEC supplement for the heterogeneity analysis.
The goal is to produce a state × year panel of **employment rates** broken down by
age group and gender,  to test whether minimum wage effects are concentrated among
specific demographic groups (teens, young adults, women).

**Input:** `data/raw/cps/cps_00002.csv.gz`

**Output:**
- `data/intermediate/cps_clean.parquet` — individual-level 
- `data/intermediate/cps_panel.parquet` — state × year × group employment rates

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent
RAW_CPS = ROOT / "data" / "raw" / "cps"
INTERMEDIATE = ROOT / "data" / "intermediate"
INTERMEDIATE.mkdir(parents=True, exist_ok=True)

CPS_FILE = RAW_CPS / "cps_00002.csv.gz"
OUTPUT_CLEAN = INTERMEDIATE / "cps_clean.parquet"
OUTPUT_PANEL = INTERMEDIATE / "cps_panel.parquet"

FIRST_YEAR = 2004
LAST_YEAR = 2024

## 1. Load and inspect the raw file

In [2]:
raw = pd.read_csv(CPS_FILE, dtype={"STATEFIP": str})

print("Shape  :", raw.shape)
print("Dtypes :")
print(raw.dtypes)
print("\nHead:")
raw.head()

Shape  : (7632720, 21)
Dtypes :
YEAR           int64
SERIAL         int64
MONTH          int64
HWTFINL      float64
CPSID          int64
ASECFLAG     float64
HFLAG        float64
ASECWTH      float64
STATEFIP         str
PERNUM         int64
WTFINL       float64
CPSIDP         int64
CPSIDV         int64
ASECWT       float64
AGE            int64
SEX            int64
EMPSTAT        int64
LABFORCE       int64
UHRSWORKT      int64
WKSWORK2     float64
INCWAGE      float64
dtype: object

Head:


,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,HFLAG,ASECWTH,STATEFIP,PERNUM,...,CPSIDP,CPSIDV,ASECWT,AGE,SEX,EMPSTAT,LABFORCE,UHRSWORKT,WKSWORK2,INCWAGE
0,2010,1,3,NaN,20091200101400,1.0,NaN,485.99,23,1,...,20091200101401,200912001014011,485.99,65,2,32,1,999,0.0,0.0
1,2010,2,3,NaN,20091201328700,1.0,NaN,531.71,23,1,...,20091201328701,200912013287011,531.71,54,1,32,1,999,4.0,12000.0
2,2010,3,3,NaN,20091202862200,1.0,NaN,474.40,23,1,...,20091202862201,200912028622011,474.40,73,2,36,1,999,0.0,0.0
3,2010,3,3,NaN,20091202862200,1.0,NaN,474.40,23,2,...,20091202862202,200912028622021,474.40,71,1,36,1,999,0.0,0.0
4,2010,4,3,NaN,20091201328800,1.0,NaN,486.65,23,1,...,20091201328801,200912013288011,486.65,46,1,10,2,30,6.0,44000.0


In [3]:
print("Year range    :", raw["YEAR"].min(), "to", raw["YEAR"].max())
print("Unique states :", raw["STATEFIP"].nunique())
print("\nMissing values:")
print(raw.isnull().sum())
print("\nASECFLAG value counts:")
print(raw["ASECFLAG"].value_counts())
print("\nLABFORCE value counts:")
print(raw["LABFORCE"].value_counts())
print("\nEMPSTAT value counts:")
print(raw["EMPSTAT"].value_counts().sort_index())

Year range    : 2010 to 2026
Unique states : 51

Missing values:
YEAR               0
SERIAL             0
MONTH              0
HWTFINL      2713615
CPSID              0
ASECFLAG     4443945
HFLAG        7433164
ASECWTH      4919105
STATEFIP           0
PERNUM             0
WTFINL       2713615
CPSIDP             0
CPSIDV             0
ASECWT       4919105
AGE                0
SEX                0
EMPSTAT            0
LABFORCE           0
UHRSWORKT          0
WKSWORK2     4919105
INCWAGE      4919105
dtype: int64

ASECFLAG value counts:
ASECFLAG
1.0    2713615
2.0     475160
Name: count, dtype: int64

LABFORCE value counts:
LABFORCE
2    3750360
1    2433793
0    1448567
Name: count, dtype: int64

EMPSTAT value counts:
EMPSTAT
0     1423585
1       24982
10    3453747
12     129321
21     152069
22      15223
32     299832
34     869569
36    1264392
Name: count, dtype: int64


## 2. Filter to March ASEC, working-age adults in the labor force

In [4]:
n_raw = len(raw)

# Keep only March ASEC observations
df = raw[raw["ASECFLAG"] == 1].copy()
print(f"After ASEC filter         : {len(df):,} (dropped {n_raw - len(df):,})")

# Analysis years only
df = df[df["YEAR"].between(FIRST_YEAR, LAST_YEAR)]
print(f"After year filter         : {len(df):,}")

# Working-age adults (16–64)
df = df[df["AGE"].between(16, 64)]
print(f"After age filter (16–64)  : {len(df):,}")

# In the labor force
df = df[df["LABFORCE"] == 2]
print(f"After labor force filter  : {len(df):,}")

After ASEC filter         : 2,713,615 (dropped 4,919,105)
After year filter         : 2,713,615
After age filter (16–64)  : 1,703,632
After labor force filter  : 1,245,354


## 3. Create analysis variables

In [5]:
# Employed = at work or has job but not at work
df["employed"] = df["EMPSTAT"].isin([10, 12]).astype(int)

# Gender
df["female"] = (df["SEX"] == 2).astype(int)

# Age groups — standard low-wage worker categories
df["age_group"] = pd.cut(
    df["AGE"],
    bins=[15, 19, 24, 34, 44, 54, 64],
    labels=["16-19", "20-24", "25-34", "35-44", "45-54", "55-64"],
)
df["age_group"] = df["age_group"].astype(str)

# Zero-pad state FIPS to 2 digits
df["state_fips"] = df["STATEFIP"].str.zfill(2)

# Year and weight
df["year"] = df["YEAR"].astype(int)
df["weight"] = df["ASECWT"].astype(float)

print("Age group distribution:")
print(df["age_group"].value_counts().sort_index())
print(
    f"\nOverall employment rate (weighted): {np.average(df['employed'], weights=df['weight']):.3f}"
)

Age group distribution:
age_group
16-19     53554
20-24    107009
25-34    279605
35-44    311291
45-54    291822
55-64    202073
Name: count, dtype: int64

Overall employment rate (weighted): 0.939


## 4. Handle NIU / missing codes

In [6]:
# UHRSWORKT: 999 = NIU
df["UHRSWORKT"] = df["UHRSWORKT"].replace(999, np.nan)

# INCWAGE: 999999 = NIU, 999998 = missing
df["INCWAGE"] = df["INCWAGE"].replace({999999: np.nan, 999998: np.nan})

print("Missing values after NIU handling:")
print(
    df[["UHRSWORKT", "INCWAGE", "employed", "female", "age_group", "weight"]]
    .isnull()
    .sum()
)

Missing values after NIU handling:
UHRSWORKT    74845
INCWAGE          0
employed         0
female           0
age_group        0
weight           0
dtype: int64


## 5. Select final columns and save individual-level file

In [7]:
KEEP_COLS = [
    "state_fips",
    "year",
    "AGE",
    "age_group",
    "female",
    "employed",
    "INCWAGE",
    "UHRSWORKT",
    "weight",
]

cps_clean = df[KEEP_COLS].copy().reset_index(drop=True)

print("Individual-level file shape:", cps_clean.shape)
cps_clean.to_parquet(OUTPUT_CLEAN, index=False)

Individual-level file shape: (1245354, 9)


## 6. Collapse to state × year × age_group × gender employment rates

We compute **weighted** employment rates for each cell. These will be merged with the
minimum wage panel in the heterogeneity notebook.

In [8]:
def weighted_mean(group):
    w = group["weight"]
    return pd.Series(
        {
            "emp_rate": np.average(group["employed"], weights=w),
            "n_individuals": len(group),
            "total_weight": w.sum(),
        }
    )


# Full breakdown: state × year × age_group × female
cps_panel = (
    cps_clean.groupby(["state_fips", "year", "age_group", "female"])
    .apply(weighted_mean)
    .reset_index()
)

# Also compute state × year × age_group (both genders combined)
cps_by_age = (
    cps_clean.groupby(["state_fips", "year", "age_group"])
    .apply(weighted_mean)
    .reset_index()
)
cps_by_age["female"] = -1  # sentinel: both genders

# Stack
cps_panel = pd.concat([cps_panel, cps_by_age], ignore_index=True)
cps_panel["n_individuals"] = cps_panel["n_individuals"].astype(int)

print("Collapsed panel shape:", cps_panel.shape)
print("\nSample:")
cps_panel.head(12)

Collapsed panel shape: (13770, 7)

Sample:


,state_fips,year,age_group,female,emp_rate,n_individuals,total_weight
0,01,2010,16-19,0,0.550488,26,48874.06
1,01,2010,16-19,1,0.653889,25,49226.93
2,01,2010,20-24,0,0.749012,49,131944.63
3,01,2010,20-24,1,0.855635,45,121006.24
4,01,2010,25-34,0,0.813321,104,223268.95
5,01,2010,25-34,1,0.871249,105,213196.56
6,01,2010,35-44,0,0.899824,130,261425.96
7,01,2010,35-44,1,0.886489,131,252566.45
8,01,2010,45-54,0,0.892222,134,278535.09
9,01,2010,45-54,1,0.925062,119,235168.33


In [9]:
# Check national employment rate by age group
national = (
    cps_clean.groupby("age_group")
    .apply(lambda g: np.average(g["employed"], weights=g["weight"]))
    .reset_index()
    .rename(columns={0: "emp_rate"})
)
print(f"National weighted employment rate by age group ({FIRST_YEAR}–{LAST_YEAR}):")
print(national.to_string(index=False))

# By gender
by_gender = (
    cps_clean.groupby("female")
    .apply(lambda g: np.average(g["employed"], weights=g["weight"]))
    .reset_index()
    .rename(columns={0: "emp_rate", "female": "is_female"})
)
print("\nNational weighted employment rate by gender:")
print(by_gender.to_string(index=False))

National weighted employment rate by age group (2004–2024):
age_group  emp_rate
    16-19  0.832942
    20-24  0.896329
    25-34  0.936785
    35-44  0.950996
    45-54  0.953411
    55-64  0.956074

National weighted employment rate by gender:
 is_female  emp_rate
         0  0.934240
         1  0.944577


## 7. Save collapsed panel

In [10]:
cps_panel.to_parquet(OUTPUT_PANEL, index=False)